# 🐳 PHASE 2: Docker, Netzwerk & Reverse Proxy (Traefik + OAuth2-Proxy)
**Projekt: Mai_AI (MaiOmni) — Reverse Proxy & Dynamic routing**

In dieser Phase schalten wir das schützende **Traefik-Gateway** und den **Google OAuth2 Proxy** vor dein System. Dies entspricht der Architektur-Spezifikation für einen sicheren Web-Zugang über DuckDNS (`mai-ai.duckdns.org`), ohne den Host angreifbar zu machen[cite: 3].

--- 
### 🎯 Ziele dieser Phase:
1. **Validierung & Orchestrierung:** Validierung des lokalen Docker-Status und Erstellung des isolierten Plattform-Netzwerks (`mai-ai_network`) sowie der Kern-Volumes[cite: 3].
2. **Gateway & Sicherheit:** Initialisierung von Traefik und des Auth-Gateways über Docker-Compose inklusive Verifizierung des sicheren Header-Verhaltens (`X-Forwarded-User`)[cite: 3].
3. **Dynamische Benutzer-Isolation (User-Spaces):** Bereitstellung getrennter Umgebungen (`/data/users/<user_id>/`) mit strikten Filesystem-Quotas, um Cross-User-Zugriffe technisch auszuschließen.
4. **Asynchrones Messaging (SQLite-Inbox):** Entkoppelte Kommunikation zwischen HTML-Oberfläche und Agenten-Engine zur sicheren Pufferung von Anfragen.
5. **Lifecycle-Management (Idle-Timeout):** Automatisches Herunterfahren (Auto-Shutdown) der Agenten-Container nach >10 Minuten Inaktivität zur Ressourceneinsparung und On-Demand-Reaktivierung bei neuem User-Input.
6. **Container Hardening:** Absicherung der Container durch Read-only Dateisysteme, entzogene Linux-Capabilities (`cap_drop: [ALL]`), Non-Root Ausführung und harte Logging-Limits.

---

### 🛠️ Schritt 1: Docker CLI & Daemon Konnektivität (Hardening-Check)
Wir prüfen mittels Python Docker SDK, ob die Docker Engine läuft und erreichbar ist. 

Im Sinne der **Anti-Gravity-Architektur** stellt dieser Schritt nicht nur die Verbindung her, sondern validiert gleichzeitig die Betriebsumgebung:
*   **Daemon-Integrität:** Sicherstellung, dass der Docker-Service auf einem gesicherten Socket kommuniziert.
*   **Ressourcen-Verfügbarkeit:** Vorab-Check der Engine-Version, um Kompatibilität mit den 2026er Container-Hardening-Features (wie `read_only=True` und `cap_drop`) zu garantieren.
*   **Orchestrierungs-Bereitschaft:** Aufbau der Verbindung, um dynamisch isolierte Benutzer-Container ("Anti-Gravity Kapseln") nach Bedarf zu starten oder zu stoppen.

In [7]:
import docker
import os
import sys
import time
import subprocess

def connect_docker(retries=3, delay=5):
    """
    Überprüft robust, ob der Docker-Daemon läuft.
    Falls nicht, wird die offizielle Docker.app (macOS), Docker Desktop (Windows) 
    oder der Systemdienst gestartet.
    Die Info-Ausgaben werden im Jupyter Notebook (.ipynb) angezeigt.
    """
    for i in range(retries):
        try:
            # 1. Schritt: Versuchen, eine Verbindung zum laufenden Daemon herzustellen
            client = docker.from_env()
            client.ping()
            print(f"[✓] Docker-Daemon ist aktiv. Server Version: {client.version()['Version']}")
            return client
        except (docker.errors.DockerException, Exception) as e:
            print(f"[!] Docker-Daemon ist aktuell nicht erreichbar (Versuch {i+1}/{retries}).")
            
            if i < retries - 1:
                print(f"[...] Starte Docker-Umgebung offiziell über das System... Bitte warten.")
                try:
                    # Start-Logik je nach Betriebssystem
                    
                    if sys.platform == "darwin":  # macOS
                        # Der offizielle, vom System unterstützte Weg über das App-Bundle.
                        subprocess.run(["open", "-a", "Docker"], check=True)
                        
                    elif sys.platform == "win32":  # Windows
                        # Sicherer Pfad über die offizielle Executable, um UAC/Admin-Rechte-Probleme
                        # beim direkten Starten des Windows-Dienstes zu umgehen.
                        program_files = os.environ.get("ProgramFiles", "C:\\Program Files")
                        docker_win_path = os.path.join(program_files, "Docker", "Docker", "Docker Desktop.exe")
                        
                        if os.path.exists(docker_win_path):
                            subprocess.Popen(
                                [docker_win_path],
                                start_new_session=True
                            )
                        else:
                            # Fallback auf den Windows-Dienst, falls der Standardpfad abweicht
                            subprocess.Popen(
                                ["net", "start", "com.docker.service"],
                                creationflags=subprocess.CREATE_NO_WINDOW,
                                start_new_session=True
                            )
                        
                    elif sys.platform.startswith("linux"):  # Linux
                        # Da 'sudo systemctl' im Notebook ohne Passworteingabe einfrieren würde,
                        # geben wir eine klare Handlungsanweisung aus und nutzen die systemd Socket-Aktivierung.
                        print("[!] HINWEIS: Docker unter Linux ist offline.")
                        print("[!] Bitte starte den Dienst im Terminal mit: 'sudo systemctl start docker'")
                        print("[...] Versuche dennoch, eine Verbindung über Socket-Activation aufzubauen...")
                        
                except Exception as start_error:
                    print(f"[X] Fehler beim offiziellen Startaufruf von Docker: {start_error}")
                
                # Dem Daemon ausreichend Zeit geben, die virtuelle Umgebung hochzufahren
                print(f"[...] Warte {delay + 3} Sekunden, damit die Engine vollständig initialisieren kann...")
                time.sleep(delay + 3)
                print(f"[...] Überprüfe Verbindung nach Startversuch erneut...\n")
            else:
                print("\n[X] Docker konnte nicht gestartet werden. Bitte öffne Docker Desktop manuell.")
                raise e

# Skript ausführen
try:
    client = connect_docker()
except Exception as final_error:
    print(f"\nAbbruch: Verbindung zu Docker fehlgeschlagen.\nDetails: {final_error}")

[✓] Docker-Daemon ist aktiv. Server Version: 29.5.3


### 🌐 Schritt 2: Infrastruktur-Netzwerk & Datenschnittstellen initialisieren
Die dynamischen Benutzer-Container kommunizieren ausschließlich über das dedizierte Brücken-Netzwerk `mai-ai_network` mit dem Traefik-Gateway.

*   **Netzwerk-Isolation:** Das `mai-ai_network` fungiert als geschlossenes Segment. Container sind nur über das Gateway erreichbar; direkte Kommunikation zwischen den Benutzer-Kapseln ist unterbunden.
*   **Persistent Storage (Datenschnittstellen):** Initialisierung der gesicherten Volumes (`mai_ai_local_models`, `mai_ai_db_data`, `mai_ai_config`). Diese Volumes fungieren als verschlüsselte Datenschnittstellen zwischen dem Host und den transienten Agenten-Containern.
*   **Vollständigkeits-Check:** Vor der Inbetriebnahme wird der Status der Infrastruktur-Komponenten geprüft, um eine konsistente Basis für den automatisierten Lebenszyklus (Auto-Shutdown/-Startup) zu gewährleisten.

### 🛠️ Prozess-Logik: Infrastruktur-Provisionierung
Um ein stabiles und sicheres Mai_AI-System zu gewährleisten, folgt der Aufbau einer festen Prozess-Kette, die sicherstellt, dass jeder Agent-Container ("Kapsel") sicher eingebettet ist:

*   **Schritt 0 (Vorbereitung):** Aufbau der persistenten Verbindung zum Docker-Daemon. Hierbei erfolgt eine automatische Prüfung, ob die Engine für die benötigten Sicherheits-Features (Kernel-Namespaces für User-Isolation) bereit ist.
*   **Schritt 1 (Kommunikation):** Erstellung des dedizierten `mai-ai_network`. Dies dient als Sicherheitssegment, in dem nur autorisierte Container (Traefik, Auth-Proxy, User-Agent) miteinander kommunizieren können.
*   **Schritt 2 (Ablage & Datenschnittstelle):** Einrichtung der dedizierten Volumes (`mai_ai_local_models`, `mai_ai_db_data`, `mai_ai_config`). Diese fungieren als isolierte Datenschnittstellen, die bei Bedarf mit Benutzer-spezifischen Overlay-Systemen erweitert werden können.
*   **Schritt 3 (Container-Orchestrierung):** Die finale Instanzierung der Container. Hierbei erfolgt die "Verdrahtung": Jeder Agent wird zwingend mit dem Netzwerk aus Schritt 1 und den Schnittstellen aus Schritt 2 verknüpft, wobei Sicherheitsrichtlinien wie `read_only=True` angewendet werden.

In [8]:
import docker
import sys

# 0. Verbindung zu Docker herstellen
try:
    client = docker.from_env()
    client.ping()
except Exception as e:
    print(f"[X] Docker läuft nicht oder ist nicht erreichbar: {e}")
    sys.exit(1)

print("-" * 50)

# ==========================================
# 1. DIE KOMMUNIKATION (Netzwerk-Prüfung)
# ==========================================
network_name = "mai-ai_network"
try:
    networks = client.networks.list(names=[network_name])
    if not networks:
        client.networks.create(network_name, driver="bridge", attachable=True)
        print(f"[✓] Netzwerk '{network_name}' wurde erfolgreich erstellt.")
    else:
        print(f"[✓] Netzwerk '{network_name}' ist bereits vorhanden.")
except Exception as e:
    print(f"[!] Fehler beim Erstellen des Netzwerks: {e}")

# ==========================================
# 2. DIE ABLAGE & DATENSCHNITTSTELLE (Volumes)
# ==========================================
volumes_to_check = ["mai_ai_local_models", "mai_ai_db_data", "mai_ai_config"]
for volume in volumes_to_check:
    try:
        client.volumes.get(volume)
        print(f"[✓] Datenschnittstelle '{volume}' ist vorhanden.")
    except docker.errors.NotFound:
        client.volumes.create(name=volume)
        print(f"[✓] Datenschnittstelle '{volume}' wurde neu angelegt.")
    except Exception as e:
        print(f"[!] Fehler bei der Datenschnittstelle {volume}: {e}")

# ==========================================
# 3. DIE CONTAINER (Status-Überprüfung)
# ==========================================
# Liste deiner geplanten Container-Namen für das System
required_containers = ["mai_ai_ollama_engine"]
for container_name in required_containers:
    try:
        container = client.containers.get(container_name)
        print(f"[✓] Container '{container_name}' existiert (Status: {container.status}).")
    except docker.errors.NotFound:
        print(f"[!] Container '{container_name}' fehlt noch (Wird im nächsten Schritt erzeugt).")
    except Exception as e:
        print(f"[!] Fehler bei der Container-Überprüfung {container_name}: {e}")

print("-" * 50)
print("[✓] Infrastruktur-Check abgeschlossen.")

--------------------------------------------------
[✓] Netzwerk 'mai-ai_network' ist bereits vorhanden.
[✓] Datenschnittstelle 'mai_ai_local_models' ist vorhanden.
[✓] Datenschnittstelle 'mai_ai_db_data' ist vorhanden.
[✓] Datenschnittstelle 'mai_ai_config' ist vorhanden.
[!] Container 'mai_ai_ollama_engine' fehlt noch (Wird im nächsten Schritt erzeugt).
--------------------------------------------------
[✓] Infrastruktur-Check abgeschlossen.


### 🏗️ Schritt 3: Infrastruktur-Gateway & Security-Proxy aktivieren
In diesem Schritt erfolgt die Aktivierung der "Sicherheitsschleuse" des Mai_AI-Systems. Wir initiieren den Start-Vorgang für Traefik (Edge-Router) und den OAuth2-Proxy (Identitäts-Validierung).

*   **Sicherheits-Gatekeeping:** Traefik fungiert als Ingress-Controller und erzwingt, dass jeder eingehende Request erst die Authentifizierung durch den OAuth2-Proxy durchlaufen muss, bevor er eine "Anti-Gravity Kapsel" erreicht.
*   **Zero-Trust-Startsequenz:** Die Ausführung der `Start_AI.py` orchestriert die Dienste so, dass sie als verschlossenes Gesamtsystem im `mai-ai_network` hochfahren. 
*   **Konfigurations-Audit:** Das System führt einen Pre-Flight-Check deiner `.env`-Konfiguration durch. Dies stellt sicher, dass alle notwendigen Identitäts-Token und Domain-Parameter korrekt gesetzt sind, bevor die Kapsel-Infrastruktur online geht.
*   **Zustands-Übergabe:** Nach erfolgreichem Start erfolgt eine Validierung des Status aller Komponenten (Traefik, Auth, Ollama-Engine), um sicherzustellen, dass das System voll einsatzbereit ist.

*Hinweis: Bitte stelle sicher, dass die `.env`-Datei lokal im Root-Verzeichnis existiert und deine spezifischen OAuth2-Zugangsdaten enthält.*

In [10]:
import os
import subprocess
import logging

# Einfaches Logging für das Notebook-Feedback
print("\n" + "="*60)
print(" 🐳 PHASE 2: ORCHESTRIERUNG & START-VALIDIERUNG")
print("="*60 + "\n")

env_path = "../.env"

# 1. Logik-Prüfung: Existiert die Konfiguration?
if not os.path.exists(env_path):
    print("[!] KONTROLLE FEHLGESCHLAGEN:")
    print("    Keine .env-Datei im Projekt-Root gefunden!")
    print("    Bitte kopiere '.env.example' zu '.env' und trage deine OAuth2-Daten ein.")
else:
    print("[✓] Umgebungskonfiguration (.env) lokalisiert.")
    
    # Optionaler Pre-Flight-Check der Variablen für das Notebook-Protokoll
    try:
        with open(env_path, "r") as f:
            env_content = f.read()
            
        print("    -> Überprüfe Funktionsbereich 1 & 2 (Ingress & Security)...")
        if "GOOGLE_CLIENT_ID" in env_content and "DOMAIN_NAME" in env_content:
            print("    [✓] OAuth2-Zugangsdaten und Domain-Konfiguration sind hinterlegt.")
        else:
            print("    [!] Warnung: Einige Schlüsselvariablen fehlen möglicherweise in der .env!")
    except Exception as e:
        print(f"    [!] Fehler beim Lesen der .env-Metadaten: {e}")

    # 2. Logik-Prüfung: Start der 3 Funktionsbereiche via Start_AI.py
    print("\n[*] Initialisiere System-Start über den AI Orchestrator...")
    try:
        # Führt das überarbeitete Start-Skript aus, das Ingress, Security und Engine koppelt
        result = subprocess.run(["python", "../src/docker_py/Start_AI.py"], check=True)
        
        print("\n" + "="*60)
        print("[✓] ERFOLG: Start_AI.py hat die Infrastruktur übergeben.")
        print("    Alle Container (Traefik, Auth, Ollama) sind im 'mai-ai_network' aktiv.")
        print("="*60)
        
    except subprocess.CalledProcessError as e:
        print(f"\n[!] FEHLER: Start_AI.py wurde mit Fehlercode {e.returncode} abgebrochen.")
        print("    Bitte überprüfe die Docker Desktop Logs der Container.")
    except Exception as e:
        print(f"\n[!] Unerwarteter Fehler beim Skriptaufruf: {e}")


 🐳 PHASE 2: ORCHESTRIERUNG & START-VALIDIERUNG

[!] KONTROLLE FEHLGESCHLAGEN:
    Keine .env-Datei im Projekt-Root gefunden!
    Bitte kopiere '.env.example' zu '.env' und trage deine OAuth2-Daten ein.


---
### 🔒 Schritt 4: Container Hardening & Sicherheits-Audit
Nachdem das Gateway und die Basisdienste laufen, validiert das System die Einhaltung der strikten Sicherheitsvorgaben für alle aktiven Container:

*   **Read-Only Dateisystem:** Kern-Container laufen im `read-only`-Modus, um Manipulationen am System-Code zu verhindern.
*   **Cap-Drop (Minimalrechte):** Allen Containern werden standardmäßig sämtliche Linux-Kernel-Privilegien entzogen (`cap_drop: [ALL]`), um Angriffsvektoren auszuschließen.
*   **Non-Root Ausführung:** Keine Prozesse laufen mit Root-Rechten im Container.
*   **Ressourcen-Limits:** Harte CPU- und RAM-Obergrenzen verhindern ein "Ausufern" einzelner Agenten-Instanzen.
---

### 🚀 Nächster Meilenstein: Benutzer-Frontend & Lifecycle-Integration
Die Infrastruktur (Traefik, Auth-Proxy, Docker-Netzwerke, Volumes) ist nun vollständig eingerichtet und abgesichert. 

Fahre im nächsten Notebook fort, um die dynamische Erstellung der Benutzer-Kapseln und die SQLite-Inbox zu implementieren:
👉 **[03_html_embed.ipynb](file:notebooks/03_html_embed.ipynb)**

---
### 📋 4. Matrix der Infrastruktur-Vollständigkeit (Mai_AI System-Audit)
Um den sicheren Web-Zugriff vom Internet (HTML-Interface) bis zum Chat-Antwort-Workflow zu gewährleisten, basieren wir auf dem folgenden Implementierungsstand:

| Funktionsbereich | Status | Geplante Erweiterung / Implementierung |
| :--- | :--- | :--- |
| **Docker Daemon** | Vorhanden | Keine erforderlich |
| **Netzwerk & Konnektivität** | Vorhanden | Keine erforderlich |
| **Daten-Persistenz** | Vorhanden | Backup-Strategie für Volumes prüfen |
| **Orchestrierung & Start** | Vorhanden | Integrierte Fehlerbehandlung |
| **Sicherheit & Ingress** | Vorhanden | Zertifikats-Management |
| **Health-Checks** | Fehlt | Warte-Logik für Container-Bereitschaft |
| **Ressourcen-Management** | Fehlt | CPU/RAM/GPU Limits für Chat-Engine |
| **Benutzer-Isolation** | **Neu: Benötigt** | Dynamische User-Volumes für Chat-Historie |
| **Inter-Container Messaging** | **Neu: Benötigt** | SQLite-Inbox für Chat-Request-Pufferung |
| **Speicher-Begrenzung** | **Neu: Benötigt** | Quotas für User-Dateien |
| **Idle-Timeout** | **Neu: Benötigt** | Auto-Shutdown bei Chat-Inaktivität |
| **Container Hardening** | **Neu: Benötigt** | Read-only & Drop Caps für Chat-Container |

---

### 🛡️ 5. Security- & Lifecycle-Regeln für das Mai_AI-System
Die noch ausstehenden Implementierungen (gekennzeichnet als *Neu: Benötigt* in der Architektur-Tabelle) greifen direkt in das Laufzeitverhalten und die Stabilität deiner Container ein:

1. **Dynamische Benutzer-Isolation:** Jeder User erhält zur Laufzeit eine isolierte Instanz, um Datenvermischungen auf Dateiebene und unbefugte Einblicke in den Chat-Verlauf konsequent auszuschließen.
2. **SQLite-Inbox-Messaging:** Anfragen von der HTML-Oberfläche werden asynchron gepuffert, während sich der entsprechende Agenten-Container im Standby befindet.
3. **Automated Idle-Lifecycle:** Überschreitet die Inaktivität der Chat-Sitzung ein Limit von 10 Minuten, fährt der Container (`container.stop()`) eigenständig herunter, um Systemressourcen auf dem Host freizugeben.
4. **Hardened Runtime:** Jede Container-Instanz wird strikt mit reduzierten Kernel-Rechten (`cap_drop: [ALL]`) gestartet, um das Gesamtsystem abzusichern.
---

### 🔄 Was kommt als Nächstes? (Ausblick & Roadmap)

Deine Gateway-Infrastruktur ist nun vollständig initialisiert und läuft im Hintergrund, um als **"Security-Schild"** unautorisierte Anfragen konsequent abzuweisen. Die Container sind erfolgreich verdrahtet, gehärtet und in das `mai-ai_network` eingebunden.

**Die nächsten Meilensteine zur systemweiten Vollständigkeit:**

1.  **Dynamische Isolation:** Implementierung der User-Space-Erzeugung (`/data/users/<user_id>/`), um jeden Benutzer in einer exklusiven Kapsel zu betreiben.
2.  **Messaging & Inbox:** Integration der SQLite-Datenbank zur asynchronen Request-Verarbeitung.
3.  **Lifecycle-Automation:** Aktivierung des Idle-Timeout-Wächters für das automatische Herunterfahren inaktiver Agenten.
4.  **Frontend-Integration:** Bau des Streamlit-User-Images und Inbetriebnahme des Dynamic Provisioners.

Fahre nun fort mit dem nächsten Knotenpunkt, um die Benutzer-Kapseln live zu provisionieren:
👉 **[03_html_embed.ipynb](file:notebooks/03_html_embed.ipynb)**